# pii-anon Quickstart

This notebook walks through the core capabilities of **pii-anon** — a Python library for detecting and transforming personally identifiable information (PII).

## Installation

```bash
pip install pii-anon            # core library
pip install pii-anon[crypto]     # adds AES-SIV tokenization
```

If you cloned from GitHub:

```bash
pip install -e .[cli,crypto]
```

## 1. Basic pseudonymization

The simplest way to use pii-anon: create an orchestrator, pass it some text, and get back a pseudonymized version with cryptographic tokens.

In [ ]:
from pii_anon import PIIOrchestrator, ProcessingProfileSpec, SegmentationPlan

orch = PIIOrchestrator(token_key="demo-secret-key")

profile = ProcessingProfileSpec(
    profile_id="quickstart",
    mode="weighted_consensus",
    transform_mode="pseudonymize",
)
segmentation = SegmentationPlan(enabled=False)

result = orch.run(
    {"text": "Please contact Alice Johnson at alice.johnson@acme.com or call 555-867-5309."},
    profile=profile,
    segmentation=segmentation,
    scope="demo",
    token_version=1,
)

print("Pseudonymized text:")
print(result["transformed_payload"]["text"])

## 2. Inspecting detections

Every result includes the detected entities, confidence scores, and an audit trail showing which engines contributed.

In [ ]:
print(f"Entities found: {len(result['ensemble_findings'])}\n")

for finding in result["ensemble_findings"]:
    print(f"  Type: {finding['entity_type']}")
    print(f"  Confidence: {finding['confidence']:.2f}")
    print(f"  Engines: {finding['engines']}")
    print()

envelope = result["confidence_envelope"]
print(f"Overall confidence: {envelope['score']:.2f}")
print(f"Risk level: {envelope['risk_level']}")

## 3. Transform modes

pii-anon supports several transformation strategies. Let's compare them side-by-side on the same input.

In [ ]:
sample = {"text": "Dr. Bob Smith (age 42) lives at 123 Oak St, Portland OR 97201."}

modes = ["pseudonymize", "anonymize", "redact"]

for mode in modes:
    p = ProcessingProfileSpec(
        profile_id=f"demo_{mode}",
        mode="weighted_consensus",
        transform_mode=mode,
    )
    r = orch.run(sample, profile=p, segmentation=segmentation, scope="demo", token_version=1)
    print(f"{mode:>15s}: {r['transformed_payload']['text']}")
    print()

## 4. Fusion strategies

When multiple detection engines are available, pii-anon fuses their results. The `mode` parameter controls how findings are combined — from high-recall union to strict intersection.

In [ ]:
fusion_modes = [
    ("union_high_recall", "Union (high recall)"),
    ("weighted_consensus", "Weighted consensus"),
    ("intersection_consensus", "Intersection (strict)"),
]

sample = {"text": "Email jane.doe@corp.io about the meeting with John at 10am."}

for mode, label in fusion_modes:
    p = ProcessingProfileSpec(
        profile_id=f"fusion_{mode}",
        mode=mode,
        transform_mode="anonymize",
    )
    r = orch.run(sample, profile=p, segmentation=segmentation, scope="demo", token_version=1)
    count = len(r["ensemble_findings"])
    print(f"{label:>28s}  →  {count} entities  →  {r['transformed_payload']['text']}")
    print()

## 5. Entity tracking (alias linking)

pii-anon automatically clusters mentions of the same identity. If "Alice", "Alice Johnson", and "AJ" refer to the same person, they receive the same cluster ID and consistent tokens.

In [ ]:
text = (
    "Alice Johnson joined the team last Monday. "
    "Alice will lead the backend rewrite. "
    "Reach her at alice.johnson@acme.com."
)

profile_tracking = ProcessingProfileSpec(
    profile_id="tracking_demo",
    mode="weighted_consensus",
    transform_mode="pseudonymize",
    entity_tracking_enabled=True,
)

r = orch.run(
    {"text": text},
    profile=profile_tracking,
    segmentation=segmentation,
    scope="tracking_demo",
    token_version=1,
)

print("Transformed:")
print(r["transformed_payload"]["text"])
print()

if r.get("link_audit"):
    print("Link audit (alias clusters):")
    for entry in r["link_audit"]:
        mention = entry.get("mention_text", "unknown")
        cluster = entry.get("cluster_id", "unknown")
        rule = entry.get("rule", "")
        print(f"  '{mention}'  ->  cluster {cluster}  (rule: {rule})")

## 6. Detection only (no transformation)

Sometimes you just want to scan text for PII without modifying it — for auditing, classification, or routing decisions.

In [ ]:
detect_result = orch.detect_only(
    {"text": "My SSN is 123-45-6789 and my card is 4111-1111-1111-1111."},
    profile=profile,
    segmentation=segmentation,
    scope="audit",
    token_version=1,
)

print(f"Found {len(detect_result['ensemble_findings'])} PII entities:\n")
for f in detect_result["ensemble_findings"]:
    print(f"  {f['entity_type']:>20s}  confidence={f['confidence']:.2f}  engines={f['engines']}")

## 7. Processing multiple records

For batch workloads, iterate over a list of payloads. Each record is processed independently but shares the same tokenization scope for consistent entity linking.

In [ ]:
records = [
    {"text": "Invoice to: Maria Garcia, maria.garcia@example.com"},
    {"text": "Ship to: 456 Elm Ave, Austin TX 78701, attn: M. Garcia"},
    {"text": "Payment from account ending 4242 processed for Maria."},
]

for i, rec in enumerate(records):
    r = orch.run(rec, profile=profile, segmentation=segmentation, scope="batch", token_version=1)
    print(f"Record {i + 1}: {r['transformed_payload']['text']}")
    print()

## 8. Configuration

Fine-tune engine weights, logging, and transform defaults through `CoreConfig`.

In [ ]:
from pii_anon.config import CoreConfig, EngineRuntimeConfig

config = CoreConfig(
    engines={
        "regex-oss": EngineRuntimeConfig(enabled=True, weight=1.0),
    },
)

custom_orch = PIIOrchestrator(token_key="custom-key", config=config)

r = custom_orch.run(
    {"text": "Contact support@example.com for help."},
    profile=profile,
    segmentation=segmentation,
    scope="custom",
    token_version=1,
)
print(r["transformed_payload"]["text"])
print(f"Execution plan engines: {r['execution_plan']['engine_ids']}")

## Next steps

- **CLI**: Run `pii-anon detect "some text"` from the terminal for quick one-off scans.
- **File processing**: Use `pii_anon.read_file()` and `pii_anon.write_results()` for CSV/JSONL batch jobs.
- **Pipeline builder**: Use `PipelineBuilder` for a fluent configuration API.
- **Crypto**: Install `pii-anon[crypto]` for AES-SIV tokenization and key rotation.
- **Benchmarks**: See the README for benchmark results comparing pii-anon against other PII tools.